# Computing Project 
## 3-Particle Quantum Mechanics

##### connecting to kernel

In [167]:
print('connected')

connected


**Importing Necessary Packages**


In [168]:
import warnings
warnings.filterwarnings('ignore')

In [169]:
import numpy as np
import scipy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.special import factorial
import pandas



**Defining Basis Vectors**

In [170]:
one_state_basis = [[0,0,0]]
num_one_state_basis = 1

**Generating n states $\psi$**

In [171]:
hyll_states = [[0,0,0], [0,0,1], [0,2,0]]
num_hyll_states = len(hyll_states)
test_kappa = 1

#### Testing making N

In [172]:
def expectation_N_value(J, K, M, kappa):
    zed = 2
    r_0_ang = 0.52917 ## Angstrom
    hy_lambda = zed /(kappa * r_0_ang)


    factorial_term = factorial((J + K + M +5), exact = True) #if this needs to be faster, set exact to be false
    wavelength_term = (1/(2*hy_lambda))**((J+K+M+6))
    coeff = (2*(np.pi)**2) / (M+2)
    fraction_sum = (1/(K+1) - 1/(K+3) - 1/(K+M+3) + 1/(K+M+5))
    
    if K%2 ==0:
        entry_value_N = factorial_term * coeff * wavelength_term * fraction_sum
    if K%2 ==1:
        entry_value_N = 0


    entry_value_N = factorial_term * coeff * wavelength_term * fraction_sum
    return entry_value_N


In [173]:
def generate_N(states, n, function):
    mat = [[0]*n for _ in range(n)]

    for i in range(n): #this will give the number of rows
        for j in range(n):  # this will give the number of cols
            J = states[i][0] + states[j][0]
            K = states[i][1] + states[j][1]
            M = states[i][2] + states[j][2]

            mat[i][j] = function(J, K, M, test_kappa)
    
    return mat


##### Displaying matrices

In [174]:
## to improve visualisation of functions
def show_matrix(matrix):
    
    return np.asarray(matrix)
        

show_matrix(generate_N(hyll_states, num_hyll_states, expectation_N_value))

array([[0.00338603, 0.00195977, 0.00035556],
       [0.00195977, 0.00142224, 0.00030183],
       [0.00035556, 0.00030183, 0.00014935]])

##### Checking Hyllaas' states

In [175]:
## this is to test with the hylleraas states
def correct_N(kappa):
    zed = 2
    r_0_ang = 0.52917 ##Angstrom
    lamb = zed /(kappa * r_0_ang)

    pre = (np.pi )**2 *(lamb)**(-6)

    correct_mat = [
        [pre                     , pre*(35/16)*(lamb)**(-1) , pre*(3/2)*(lamb)**(-2) ],
        [pre*(35/16)*(lamb)**(-1), pre*(6)*(lamb)**(-2)     , pre*(77/16)*(lamb)**(-3)],
        [pre*(3/2)*(lamb)**(-2)  , pre*(77/16)*(lamb)**(-3) ,pre*(9)*(lamb)**(-4)]
    ]
    
    return correct_mat


N_mat_hyll = generate_N(hyll_states, num_hyll_states, expectation_N_value)
correct_hyll = correct_N(test_kappa)
for i in range(num_hyll_states):
    for j in range(num_hyll_states):
        assert "%.5f" % N_mat_hyll[i][j] == "%.5f" % correct_hyll[i][j]

##### Generating $\beta$ 

In [176]:
N_mat = N_mat_hyll

In [177]:
def find_eig(matrix):
    values, vectors = sc.linalg.eig(matrix)
    values.sort()
    return [values, vectors]

print(find_eig(N_mat)[0])
print()
print(find_eig(N_mat)[1])


[6.83119256e-05+0.j 2.45860818e-04+0.j 4.64344512e-03+0.j]

[[ 0.84557232  0.51194807  0.15138235]
 [ 0.52400829 -0.74167385 -0.41873048]
 [ 0.10209193 -0.43339251  0.89540391]]


#### Construct $\beta$ from these eigenvalues

In [178]:
##this will give a list with the diagonal matrix and the matrix of eigenvectors
def generate_eig_mats(matrix, n):
    eig_vals, eig_vects = find_eig(matrix)
    diag = [[0]*n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i==j:
                diag[i][j]= float(eig_vals[i])
    
    return [diag, eig_vects]


beta, Y_mat = generate_eig_mats(N_mat, num_hyll_states)

Y_mat_tran = Y_mat.transpose()



In [179]:
show_matrix(beta)

array([[6.83119256e-05, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 2.45860818e-04, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 4.64344512e-03]])

####  Construct $\beta^{\frac{1}{2}}$

In [180]:
def raise_mat_to_power(matrix, power):
    
    length = len(matrix)
    assert length == len(matrix[0]) #checks square matrix

    new_mat = [[0]* length for _ in range(length)]

    for i in range(length):
        for j in range(length):
            if matrix[i][j]:
                new_mat[i][j] = (matrix[i][j])**power
    
    return new_mat


In [181]:
test_mat = [
    [1, 2, 3], 
    [2, 3, 4], 
    [1, 4, 3]
]

print(raise_mat_to_power(test_mat, 2))

[[1, 4, 9], [4, 9, 16], [1, 16, 9]]


In [182]:
beta_half_p = raise_mat_to_power(beta, 0.5)
beta_half_m = raise_mat_to_power(beta, -0.5)
show_matrix(beta_half_p)

array([[0.0082651 , 0.        , 0.        ],
       [0.        , 0.01567995, 0.        ],
       [0.        , 0.        , 0.06814283]])

#### Generating Hamiltonian

#### Generating P matrix

In [183]:
N_mat = np.asarray(N_mat)
beta = np.asarray(beta)
beta_half_p = np.asarray(beta_half_p)
beta_half_m = np.array(beta_half_m)

print(f'N_mat is a {type(N_mat)}')
print(f'Y is a {type(Y_mat)}')
print(f'Y_T is a {type(Y_mat_tran)}')
print(f'beta is a {type(beta)}')
print(f'beta_1/2 is a {type(beta_half_p)}')
print(f'beta_-1/2 is a {type(beta_half_m)}')

matrices = [N_mat, Y_mat, Y_mat_tran, beta, beta_half_m, beta_half_p]

for mat in matrices:
    print(mat.shape)



N_mat is a <class 'numpy.ndarray'>
Y is a <class 'numpy.ndarray'>
Y_T is a <class 'numpy.ndarray'>
beta is a <class 'numpy.ndarray'>
beta_1/2 is a <class 'numpy.ndarray'>
beta_-1/2 is a <class 'numpy.ndarray'>
(3, 3)
(3, 3)
(3, 3)
(3, 3)
(3, 3)
(3, 3)


In [184]:
print(np.asarray(N_mat))

[[0.00338603 0.00195977 0.00035556]
 [0.00195977 0.00142224 0.00030183]
 [0.00035556 0.00030183 0.00014935]]


In [199]:
def construct_W(states, n):
    mat = [[0]*n for _ in range(n)]
    
    for i in range(n): #this will give the number of rows
        for j in range(n):  # this will give the number of cols
            j_1 = states[i][0]  
            j_2 = states[j][0]
            k_1 = states[i][1] 
            k_2 = states[j][1]
            m_1 = states[i][2] 
            m_2 = states[j][2]

            J = j_1 + j_2
            K = k_1 + k_2
            M = m_1 + m_2


            value = (np.exp(2))*(generate_N(hyll_states, num_hyll_states, expectation_N_value)[i][j])
            mat[i][j] = value
    
    return mat

In [200]:
W_mat = np.asarray(
    construct_W(hyll_states, num_hyll_states)
)
print(W_mat)
    


[[0.02501958 0.01448083 0.00262725]
 [0.01448083 0.01050901 0.00223021]
 [0.00262725 0.00223021 0.00110353]]


#### Checking matrix W